In [1]:
import sys
sys.path.append("..")  # agar folder src bisa diimpor dari folder notebooks

import os
import pandas as pd
from src.folds import ambil_fold
from src.train import latih_svm_satu_fold
from src.evaluate import hitung_metrik

# Baca data hasil preprocessing dan tabel pembagian fold
df = pd.read_csv("../data/processed/emotweetid_preprocessed.csv")
tabel_fold = pd.read_csv("../data/processed/folds.csv")
os.makedirs("../reports/tables", exist_ok=True)

hasil_per_fold = []
semua_prediksi = []

for f in range(5):
    train, val, test = ambil_fold(df, tabel_fold, f)
    prediksi, c_terbaik, skor_val = latih_svm_satu_fold(train, val, test)

    # Hitung semua metrik pada data uji fold ini
    metrik = hitung_metrik(test["label"], prediksi)
    metrik.update({"model": "SVM+TF-IDF", "fold": f, "c_terbaik": c_terbaik,
                   "macro_f1_val": skor_val})
    hasil_per_fold.append(metrik)

    # Simpan prediksi untuk confusion matrix nanti
    semua_prediksi.append(pd.DataFrame({
        "fold": f, "label_asli": test["label"], "prediksi": prediksi
    }))
    print(f"Fold {f}: C={c_terbaik}, Macro F1 val={skor_val:.3f}, Macro F1 uji={metrik['macro_f1']:.3f}")

# Simpan tabel hasil per fold dan semua prediksi
tabel_hasil = pd.DataFrame(hasil_per_fold)
tabel_hasil.to_csv("../reports/tables/svm_per_fold.csv", index=False)
pd.concat(semua_prediksi).to_csv("../reports/tables/svm_prediksi.csv", index=False)

# Ringkasan: rata-rata dan simpangan baku antar fold
print("\nRingkasan 5 fold (rata-rata +- simpangan baku):")
for kolom in ["macro_f1", "accuracy", "weighted_f1"]:
    print(f"{kolom:12s}: {tabel_hasil[kolom].mean():.3f} +- {tabel_hasil[kolom].std():.3f}")

Fold 0: C=1, Macro F1 val=0.852, Macro F1 uji=0.819
Fold 1: C=1, Macro F1 val=0.820, Macro F1 uji=0.846
Fold 2: C=0.1, Macro F1 val=0.888, Macro F1 uji=0.846
Fold 3: C=1, Macro F1 val=0.884, Macro F1 uji=0.813
Fold 4: C=0.1, Macro F1 val=0.820, Macro F1 uji=0.851

Ringkasan 5 fold (rata-rata +- simpangan baku):
macro_f1    : 0.835 +- 0.017
accuracy    : 0.832 +- 0.017
weighted_f1 : 0.831 +- 0.017


In [2]:
import numpy as np

rng = np.random.default_rng(0)  # pengacak dengan seed tetap
skor_acak = []

for f in range(5):
    train, val, test = ambil_fold(df, tabel_fold, f)
    train, val = train.copy(), val.copy()

    # Acak label HANYA di data latih dan validasi; label data uji dibiarkan asli
    train["label"] = rng.permutation(train["label"].to_numpy())
    val["label"] = rng.permutation(val["label"].to_numpy())

    prediksi, c_terbaik, _ = latih_svm_satu_fold(train, val, test)
    skor = hitung_metrik(test["label"], prediksi)["macro_f1"]
    skor_acak.append(skor)
    print(f"Fold {f}: Macro F1 uji (label latih diacak) = {skor:.3f}")

print(f"\nRata-rata: {np.mean(skor_acak):.3f} +- {np.std(skor_acak, ddof=1):.3f}")

Fold 0: Macro F1 uji (label latih diacak) = 0.167
Fold 1: Macro F1 uji (label latih diacak) = 0.153
Fold 2: Macro F1 uji (label latih diacak) = 0.155
Fold 3: Macro F1 uji (label latih diacak) = 0.134
Fold 4: Macro F1 uji (label latih diacak) = 0.158

Rata-rata: 0.153 +- 0.012
